In [1]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import numpy as np

creds, _ = default()
gc = gspread.authorize(creds)

SHEET_ID = "1XlT5aOjEMJu1q79yGORLiBd9msf5vh1M5pRaE0v2l4U"

sh = gc.open_by_key(SHEET_ID)

worksheet = sh.worksheet("Master_Dataset")

datos = worksheet.get_all_records(value_render_option='FORMATTED_VALUE')

df_original = pd.DataFrame(datos)

In [2]:
df_original = df_original.copy()

df_original['xG'] = df_original['xG']/1000

In [3]:
#Veo que todos los tipos de valores sean correctos

df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Fecha_Torneo             202 non-null    int64  
 1   Rival                    202 non-null    object 
 2   Equipo                   202 non-null    object 
 3   Condicion                202 non-null    object 
 4   Jugador                  202 non-null    object 
 5   Posicion                 202 non-null    object 
 6   Minutos_Jugados          202 non-null    int64  
 7   Puntos_Opta              202 non-null    int64  
 8   Ranking                  202 non-null    int64  
 9   Goles                    202 non-null    int64  
 10  Asistencias              202 non-null    int64  
 11  xG                       202 non-null    float64
 12  Tiros_Totales            202 non-null    int64  
 13  Tiros_Arco               202 non-null    int64  
 14  Tiros_Desviados          2

In [24]:
#Agrupo haciendo una sumatoria de las columnas que son de mi interés para el análisis

columnas_a_sumar = [
    'Minutos_Jugados', 'xG', 'Goles', 'Situaciones_Creadas_SCA',
    'Centros', 'Tiros_Totales', 'Tiros_Arco', 'Tiros_Dentro_Area', 'Tiros_Cabeza'
]

df_totales = df_original.groupby('Jugador')[columnas_a_sumar].sum().reset_index()

In [25]:
#Normalizo los datos por para comparar de manera justa el rendimiento, sin importar si
#un futbolista jugó un partido completo o entró desde el banco

df_totales['xG_p90'] = ((df_totales['xG']/df_totales['Minutos_Jugados'])*90).round(2)

df_totales['Situaciones_Creadas_SCA_p90'] = ((df_totales['Situaciones_Creadas_SCA']/df_totales['Minutos_Jugados'])*90).round(2)

df_totales['Tiros_Totales_p90'] = ((df_totales['Tiros_Totales']/df_totales['Minutos_Jugados'])*90).round(2)

df_totales['Centros_p90'] = ((df_totales['Centros']/df_totales['Minutos_Jugados'])*90).round(2)

df_totales['Tiros_Arco_p90'] = ((df_totales['Tiros_Arco']/df_totales['Minutos_Jugados'])*90).round(2)

In [26]:
# Aplico un filtro booleano para quedarme con los jugadores con 300 o más minutos para que los que jugaron poco no distorsionen el análisis

df_filtrado = df_totales[df_totales['Minutos_Jugados'] >= 300].copy()

In [27]:
#Ahora quiero ver quién es el verdadero generador de peligro del equipo a lo largo de estas 13 fechas
#Y qué peso tiene en la totalidad del equipo

#En primer lugar calculo las situaciones creadas TOTALES del equipo, incluyendo a todos los jugadores sin importar los minutos jugados

total_sca_equipo = df_totales['Situaciones_Creadas_SCA'].sum()

# Luego, paso a determinar el peso individual sobre el universo total

df_filtrado['Pct_aporte_SCA'] = ((df_filtrado['Situaciones_Creadas_SCA'] / total_sca_equipo) * 100).round(2)

df_filtrado[['Jugador','Situaciones_Creadas_SCA','Situaciones_Creadas_SCA_p90','Pct_aporte_SCA']].sort_values(by='Pct_aporte_SCA', ascending=False)

,Jugador,Situaciones_Creadas_SCA,Situaciones_Creadas_SCA_p90,Pct_aporte_SCA
6,G. Cerato,21,1.94,15.44
0,A. Luna,19,1.75,13.97
13,J. Lázaro,19,2.62,13.97
2,D. Sosa,19,1.46,13.97
20,M. Tissera,9,0.93,6.62
5,G. Abregú,8,0.66,5.88
18,M. Gallardo,7,0.61,5.15
3,F. Alarcón,5,0.45,3.68
11,J. Córdoba,5,0.74,3.68
12,J. Galván,4,0.34,2.94


In [28]:
#Ya que tenemos los que más peligro generar, vamos a ver por qué vías lo realizan, ya sea centros o combinaciones por el medio

df_filtrado[['Jugador', 'Situaciones_Creadas_SCA_p90', 'Pct_aporte_SCA', 'Centros_p90']].sort_values(by='Pct_aporte_SCA', ascending=False)

,Jugador,Situaciones_Creadas_SCA_p90,Pct_aporte_SCA,Centros_p90
6,G. Cerato,1.94,15.44,5.73
0,A. Luna,1.75,13.97,7.00
13,J. Lázaro,2.62,13.97,5.51
2,D. Sosa,1.46,13.97,5.15
20,M. Tissera,0.93,6.62,0.31
5,G. Abregú,0.66,5.88,0.00
18,M. Gallardo,0.61,5.15,0.88
3,F. Alarcón,0.45,3.68,0.18
11,J. Córdoba,0.74,3.68,2.67
12,J. Galván,0.34,2.94,0.78


In [29]:
df_filtrado['Pct_Punteria'] = np.where(
    df_filtrado['Tiros_Totales'] > 0,
    ((df_filtrado['Tiros_Arco'] / df_filtrado['Tiros_Totales']) * 100).round(1),
    0
)

columnas_tiro = [
    'Jugador', 'Goles', 'Tiros_Totales_p90', 'Tiros_Arco_p90',
    'Pct_Punteria', 'Tiros_Dentro_Area', 'Tiros_Cabeza'
]

df_filtrado[columnas_tiro].sort_values(by='Tiros_Totales_p90', ascending=False)

,Jugador,Goles,Tiros_Totales_p90,Tiros_Arco_p90,Pct_Punteria,Tiros_Dentro_Area,Tiros_Cabeza
0,A. Luna,2,3.87,1.29,33.3,19,1
13,J. Lázaro,2,2.21,0.41,18.8,11,2
20,M. Tissera,1,2.07,0.93,45.0,15,2
6,G. Cerato,4,1.20,0.74,61.5,9,0
2,D. Sosa,0,1.08,0.38,35.7,9,1
18,M. Gallardo,1,1.05,0.18,16.7,1,0
11,J. Córdoba,0,1.04,0.30,28.6,5,0
15,L. Mosevich,2,0.77,0.31,40.0,10,6
3,F. Alarcón,3,0.72,0.27,37.5,7,5
12,J. Galván,1,0.52,0.09,16.7,5,3


In [30]:
# Ver si están por encima o por debajo de lo que marcaban las probabilidades
df_filtrado['Delta_Goles_xG'] = (df_filtrado['Goles'] - df_filtrado['xG']).round(2)

columnas_final = ['Jugador', 'Goles', 'xG', 'Delta_Goles_xG', 'Pct_Punteria']
df_filtrado[columnas_final].sort_values(by='Goles', ascending=False)

,Jugador,Goles,xG,Delta_Goles_xG,Pct_Punteria
6,G. Cerato,4,0.747,3.25,61.5
3,F. Alarcón,3,0.733,2.27,37.5
13,J. Lázaro,2,1.274,0.73,18.8
0,A. Luna,2,2.103,-0.10,33.3
15,L. Mosevich,2,1.179,0.82,40.0
18,M. Gallardo,1,0.360,0.64,16.7
20,M. Tissera,1,3.706,-2.71,45.0
12,J. Galván,1,0.359,0.64,16.7
5,G. Abregú,0,0.333,-0.33,25.0
2,D. Sosa,0,1.449,-1.45,35.7


In [32]:
# Lista de los 5 jugadores que graficaste
jugadores_grafico = ['J. Lázaro', 'G. Cerato', 'A. Luna', 'D. Sosa', 'M. Tissera']

# Chequeo directo de las dos métricas del gráfico de barras
check_barras = df_filtrado[df_filtrado['Jugador'].isin(jugadores_grafico)][
    ['Jugador', 'Situaciones_Creadas_SCA_p90', 'Centros_p90']
].sort_values(by='Situaciones_Creadas_SCA_p90', ascending=False)

print("--- AUDITORÍA GRÁFICO 1 (BARRAS) ---")
print(check_barras)

# Chequeo directo de las variables del scatter plot
check_scatter = df_filtrado[df_filtrado['Jugador'].isin(jugadores_grafico)][
    ['Jugador', 'Goles', 'xG', 'Delta_Goles_xG']
].sort_values(by='Goles', ascending=False)

print("\n--- AUDITORÍA GRÁFICO 2 (SCATTER PLOT) ---")
print(check_scatter)

--- AUDITORÍA GRÁFICO 1 (BARRAS) ---
       Jugador  Situaciones_Creadas_SCA_p90  Centros_p90
13   J. Lázaro                         2.62         5.51
6    G. Cerato                         1.94         5.73
0      A. Luna                         1.75         7.00
2      D. Sosa                         1.46         5.15
20  M. Tissera                         0.93         0.31

--- AUDITORÍA GRÁFICO 2 (SCATTER PLOT) ---
       Jugador  Goles     xG  Delta_Goles_xG
6    G. Cerato      4  0.747            3.25
0      A. Luna      2  2.103           -0.10
13   J. Lázaro      2  1.274            0.73
20  M. Tissera      1  3.706           -2.71
2      D. Sosa      0  1.449           -1.45
